In [4]:
# %% [markdown]
# # SECOM 불량 예측 — 결측 처리부터 모델 선정까지 (정석 통합본)
#
# ## 분석 철학
# 1. 누수 차단 최우선: 맨 처음 test 격리. 모든 결정(열 제거/대치/임계값)은 train만 보고,
#    test는 마지막에 딱 한 번 일반화 성능 측정에만 사용.
# 2. 개발은 cross_val로: 불량이 약 100개로 적어 단일 평가는 불안정. StratifiedKFold로
#    여러 번 평가해 안정성 확보.
# 3. 유행 기법 무비판 수용 금지: 시도 -> 효과 측정 -> 효과 없으면 기각.
#
# ## 시행착오 (아래 코드에 논리로 반영)
# - 결측 0 치환 -> 0값 없는 열에 가짜 0 무더기 -> 기각, median
# - ffill -> 자기상관 검토: 일부 열만 강하고 대부분 약함(혼재) -> 전면 적용 기각, median 통일
# - SMOTE -> 새 구조에서 재검증(셀 12.5) -> 효과 미미하면 기각, 임계값으로 해결
# - 임계값 0.5 -> 불균형이라 F1=0 -> train에서 최적 임계값 결정
# - feature selection -> 차원 축소가 핵심 레버 (상위 50개)
# - 모델 교체(Logistic 등) -> RF 못 넘음 -> RF 유지
#
# ## 평가 지표: 재현율(놓치지 않기)·정밀도·F1 (불균형이라 정확도는 무의미)
#
# ## 사용법: 이 파일 하나만 위에서 아래로 실행. (Restart Kernel and Run All 권장)
#           SMOTE 셀(12.5)은 imbalanced-learn 필요. 없으면 그 셀만 건너뜀(자동 안내).

# %% 1. 라이브러리
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve, f1_score, recall_score, precision_score

# %% 2. 데이터 로딩 + 라벨 정리
# 원본 라벨 pass=-1, fail=1 -> 직관/모델 호환 위해 fail=1, pass=0 으로 통일.
df = pd.read_csv('uci-secom.csv')
y_all = df['Pass/Fail'].replace({-1: 0, 1: 1})
X_all = df.drop(columns=['Time', 'Pass/Fail'], errors='ignore')
print("전체:", X_all.shape, "| 불량 비율:", round((y_all == 1).mean(), 3))

# %% 3. ★ 맨 처음 test 격리 (누수 차단의 출발점)
# stratify: 불량이 약 7%로 적어 무작위 분할 시 비율이 틀어진다. 층화로 양쪽 비율 동일 유지.
# 이후 X_test, y_test 는 맨 끝(셀 13) 한 번을 제외하고 절대 보지 않는다.
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.20, random_state=42, stratify=y_all
)
print(f"train 불량 {(y_train==1).mean():.3f} | test 불량 {(y_test==1).mean():.3f}  (동일해야 정상)")

# %% 4. [train만] 제거 결정 ① 상수 / 전부 NaN
# 값이 안 변하면 정보 0. 결측과 무관하게 먼저 제거. (무엇을 버릴지는 train만 보고 결정)
nuniq = X_train.nunique(dropna=True)
drop_const = nuniq[nuniq <= 1].index.tolist()
print(f"① 상수/전부NaN: {len(drop_const)}개")

# %% 4.5 [도메인 관점] 결측이 '무작위'인가 '패턴'인가
# 통계로만 자르지 않는다. 반도체 공정에서 센서 결측은 무작위가 아닌 경우가 많다:
#   설비 점검 중 / 특정 레시피에서만 미측정 / 센서 고장으로 한 구간 통째로 결측.
# 결측이 행(로트) 방향으로 뭉쳐 있으면, 결측 자체가 설비·레시피 정보(신호)일 수 있다.
# 따라서 제거 전에 (1) 행별 결측 분포 (2) 결측의 행 방향 군집성을 확인한다.
row_na = X_train.isnull().mean(axis=1)   # 각 로트(행)의 결측 비율
print("[결측의 행 방향 분포]")
print(f"  로트별 결측률 평균 {row_na.mean():.3f} | 최대 {row_na.max():.3f} | 최소 {row_na.min():.3f}")
print(f"  결측률 30% 초과 로트: {(row_na > 0.3).sum()}개 / {len(row_na)}개")

# 결측이 특정 로트에 몰리는지: 결측률 상위 로트가 전체 평균보다 두드러지면 '구간 결측' 의심
hi = row_na[row_na > row_na.quantile(0.95)]
print(f"  상위 5% 로트의 결측률: 평균 {hi.mean():.3f} (전체 평균 대비 "
      f"{hi.mean()/row_na.mean():.1f}배)")

# 결측 '여부'가 불량과 연관되는지 — 열 단위로 빠르게 스캔(결측이 신호인지 1차 점검)
miss_col = X_train.isnull().mean()
signal_like = []
for c in miss_col[(miss_col > 0.1) & (miss_col < 0.9)].index:
    present = X_train[c].notnull()
    gap = abs((y_train[present] == 1).mean() - (y_train[~present] == 1).mean())
    if gap >= 0.05:                       # 결측 여부로 불량률이 5%p 이상 갈리면
        signal_like.append((c, gap))
print(f"  결측 여부가 불량과 연관 의심되는 열: {len(signal_like)}개")
if signal_like:
    top = sorted(signal_like, key=lambda x: -x[1])[:3]
    print(f"    상위 예: {[(c, round(g,3)) for c,g in top]}")
print("해석: 군집성↑ 또는 연관 열↑ 이면 단순 제거 대신 '결측 플래그' 보존을 검토.")
print("      (이번 데이터는 익명화라 정체 불명 — 패턴만 확인하고 제거 결정에 반영)")

# %% 5. [train만] 제거 결정 ② 고결측 열 (컷오프 50%)
# 시행착오: 30%면 빡빡(많이 버림), 50%면 관대. 30~50% 열은 실측 과반이라 채울 가치 있음 -> 50%.
#           50% 초과는 채운 값이 데이터를 지배해 신뢰도 낮음 -> 제거.
miss = X_train.isnull().mean()
CUT = 0.5
high_na = miss[miss > CUT].index.tolist()

# 점검: 경계(50~70%) 열의 '결측 여부'가 불량과 관련되면 버리지 말고 플래그로 보존해야 함.
gray = miss[(miss > CUT) & (miss <= 0.7)].index
for c in gray:
    present = X_train[c].notnull()
    gap = abs((y_train[present] == 1).mean() - (y_train[~present] == 1).mean())
    if gap >= 0.05:
        print(f"  ⚠ {c}: 결측이 불량과 관련(gap {gap:.2f}) — 플래그 검토")
print(f"② 고결측(>{CUT:.0%}): {len(high_na)}개")

# %% 6. [train만] 제거 결정 ③ 중복(고상관) 열
# 상관 0.9 초과 = 사실상 같은 신호(다중공선성). 짝 중 하나만 남김. (상관도 train만 계산)
tmp = X_train.drop(columns=drop_const + high_na)
tmp = tmp.fillna(tmp.median())
corr = tmp.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
drop_corr = [c for c in upper.columns if (upper[c] > 0.9).any()]
print(f"③ 고상관(>0.9): {len(drop_corr)}개")

# %% 7. 제거 목록을 train / test 양쪽에 동일 적용
# 버릴 열은 train에서 정했고, test에선 같은 열을 따라 제거할 뿐(test는 선택에 관여 안 함).
drop_all = list(set(drop_const + high_na + drop_corr))
X_train = X_train.drop(columns=drop_all)
X_test = X_test.drop(columns=drop_all)
print(f"피처: {X_all.shape[1]} → {X_train.shape[1]}  (총 {len(drop_all)}개 제거)")

# %% 8. [근거] 대치 방법 — median 채택
# (가) 왜도: 평균은 한쪽 꼬리 극단값에 끌려가 왜곡. 치우친 열이 많으면 median이 안정적.
need = X_train.columns[X_train.isnull().any()]
skew = X_train[need].skew()
print(f"왜도 |>1 (치우침): {(skew.abs() > 1).sum()} / {len(need)}개  → median 근거")

# (나) 시행착오 — ffill 검토 후 기각: 인접 행 자기상관(lag-1)이 강해야 ffill이 정당.
#   자기상관을 실제로 계산해보니, 대부분 열은 자기상관이 약하고 '일부 열만' 강했다.
#   즉 시계열성이 열마다 혼재 → 전면 ffill은 근거 부족, 열마다 분기하면 복잡·불안정.
#   따라서 ffill은 적용하지 않고 median으로 통일한다. (실제 대치 = median만)
ac = pd.Series([X_train[c].dropna().autocorr(lag=1) for c in need], index=need).dropna()
print(f"자기상관 평균 {ac.mean():.3f}, |0.5| 초과 {(ac.abs()>0.5).sum()}개 "
      f"→ 시계열성 약함/혼재, ffill 기각 · median 채택")
# [도메인 단서] 이 결정은 '익명 데이터에서 통계로만' 내린 것. 현업에선 달라질 수 있다:
#   · 센서가 연속 공정의 측정값이면 직전 값 유지(ffill)가 타당
#   · 결측이 '공정 미진행'을 뜻하면 0 또는 별도 플래그가 타당
#   → 대치 방법 자체가 공정 의미에 따라 재검토되는 결정임.

# %% 9. 평가 도구 — 파이프라인 + 교차검증
# 파이프라인 [median 대치 → 스케일링 → 모델]: cross_val 매 fold에서 train 조각만 보고 학습(누수 차단).
# 지표: 최적 임계값에서의 F1·재현율·정밀도.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_cv(Xin, model, label):
    pipe = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), model)
    proba = cross_val_predict(pipe, Xin, y_train, cv=cv, method='predict_proba')[:, 1]
    prec, rec, th = precision_recall_curve(y_train, proba)
    f1s = 2 * prec * rec / (prec + rec + 1e-9)
    b = f1s.argmax()
    print(f"{label:18s} | 최적임계 {th[b]:.2f} | F1 {f1s[b]:.3f} "
          f"| 재현율 {rec[b]:.3f} | 정밀도 {prec[b]:.3f}")
    return f1s[b], th[b]

# %% 10. 시행착오 — 불균형 처리(임계값 vs SMOTE)
# 별도 2x2 실험에서 F1=0의 주범은 불균형이 아니라 '임계값'(0.5가 7:93엔 너무 높은 칼)이었음.
# 임계값 조정만으로 F1 0→0.25, SMOTE 추가 효과는 +0.02뿐. 단, 구조가 바뀌었으니 셀 12.5에서 재검증.

# %% 11. 베이스라인 + Feature Selection (핵심 레버)
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                            random_state=42, n_jobs=-1)
print("[전체 피처]")
evaluate_cv(X_train, rf, "RF(전체)")

# RF 중요도로 상위 K개 선택. (한계: 중요도를 train 전체로 산출 -> cross_val 점수에 약한 누수.
#  탐색용이며, test는 선택에 관여 안 하므로 최종 test 평가의 일반화 측정은 정직함.)
imp = pd.Series(
    rf.fit(X_train.fillna(X_train.median()), y_train).feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print("[상위 K개]")
for k in [20, 50, 100]:
    evaluate_cv(X_train[imp.head(k).index], rf, f"RF(상위{k})")
# 경험상 상위 50개 정점, 이상은 노이즈 피처가 성능을 깎음 -> K=50.
TOPK = 50
top_cols = imp.head(TOPK).index.tolist()

# %% 12. 시행착오 — 모델 교체 (왜 RF 유지)
print("[모델 비교 — 상위 50개]")
evaluate_cv(X_train[top_cols], rf, "RandomForest")
evaluate_cv(X_train[top_cols],
            LogisticRegression(class_weight='balanced', max_iter=1000), "LogisticReg")
# XGBoost도 비교했으나 RF 못 넘음(AUC는 RF, F1은 XGB 근소). 향상 미미 -> 해석 쉬운 RF 유지.
# selection bias 점검: '전체 피처'로도 RF 1위 -> 편향만의 결과 아님.

# %% 12.5 [재검증] 새 구조(피처50 + test격리)에서 SMOTE 효과 다시 확인
# 구조가 바뀌었으니 예전 'SMOTE 효과 미미' 결론이 유지되는지 직접 재실험.
# SMOTE는 imblearn 파이프라인 안에서 fold의 train 조각에만 적용됨(검증 조각엔 절대 X).
try:
    from imblearn.pipeline import make_pipeline as imb_make_pipeline
    from imblearn.over_sampling import SMOTE

    def evaluate_smote(Xin, use_smote, label):
        steps = [SimpleImputer(strategy='median'), StandardScaler()]
        if use_smote:
            steps.append(SMOTE(random_state=42))
        steps.append(RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
        pipe = imb_make_pipeline(*steps)
        proba = cross_val_predict(pipe, Xin, y_train, cv=cv, method='predict_proba')[:, 1]
        prec, rec, th = precision_recall_curve(y_train, proba)
        f1s = 2*prec*rec/(prec+rec+1e-9); b = f1s.argmax()
        print(f"{label:18s} | 최적임계 {th[b]:.2f} | F1 {f1s[b]:.3f} "
              f"| 재현율 {rec[b]:.3f} | 정밀도 {prec[b]:.3f}")

    print("[SMOTE 재검증 — 상위 50개]")
    evaluate_smote(X_train[top_cols], use_smote=False, label="기본")
    evaluate_smote(X_train[top_cols], use_smote=True,  label="SMOTE")
    # 차이가 작으면 'SMOTE 무익' 재확인 -> 기각 유지. 크면 결론 수정.
except ImportError:
    print("[SMOTE 재검증 건너뜀] imbalanced-learn 미설치.")
    print("  설치: 새 셀에서  import sys; !{sys.executable} -m pip install imbalanced-learn")
    print("  설치 후 커널 재시작(Restart Kernel) → Run All.")

# %% 13. ★ 최종 검증 — 격리했던 test로 딱 한 번 평가
# 지금까지 모든 결정은 train만 보고 내렸다. 이제 처음 격리한 test로 일반화 성능 측정.
final_pipe = make_pipeline(
    SimpleImputer(strategy='median'), StandardScaler(),
    RandomForestClassifier(n_estimators=200, class_weight='balanced',
                           random_state=42, n_jobs=-1)
)
# (1) train에서 최적 임계값 확정 (test는 아직 안 봄)
proba_tr = cross_val_predict(final_pipe, X_train[top_cols], y_train,
                             cv=cv, method='predict_proba')[:, 1]
prec, rec, th = precision_recall_curve(y_train, proba_tr)
f1s = 2 * prec * rec / (prec + rec + 1e-9)
THRESH = th[f1s.argmax()]

# [일관성 점검] 임계값을 F1로 정했는데, 핵심 평가지표(MCC)로 정하면 달라지나?
#   지표(MCC=채점)와 최적화 기준(F1=임계값 결정)은 다른 층이지만,
#   둘이 임계값을 크게 다르게 만들면 선택 근거를 밝혀야 한다. 그래서 비교한다.
from sklearn.metrics import matthews_corrcoef as _mcc
grid = np.linspace(0.05, 0.60, 56)
f1_th  = grid[np.argmax([f1_score(y_train, (proba_tr >= t).astype(int)) for t in grid])]
mcc_th = grid[np.argmax([_mcc(y_train, (proba_tr >= t).astype(int)) for t in grid])]
print(f"임계값 비교 — F1 기준 {f1_th:.3f} | MCC 기준 {mcc_th:.3f} "
      f"(차이 {abs(f1_th - mcc_th):.3f})")
print(f"train에서 정한 임계값(F1 기준): {THRESH:.3f}")
print("  → 두 기준의 임계값이 가까우면 지표 선택이 임계값을 크게 바꾸지 않음(안정적).")
print("     멀면 목적(재현율↑=불량 유출 방지)에 맞춰 낮은 쪽을 택하는 근거가 됨.")

# (2) train 전체로 최종 모델 학습 → 격리 test에 적용 (test 첫 사용)
final_pipe.fit(X_train[top_cols], y_train)
proba_te = final_pipe.predict_proba(X_test[top_cols])[:, 1]

# (2.1) [임계값 결정] 핵심 지표(MCC)로 최적화 시도 → 과적합 확인 → trade-off로 결론
#  ① 원칙: MCC가 핵심 평가지표라면 임계값도 MCC로 정하는 게 일관적이다.
#  ② 시도: train에서 MCC 최대 임계값 = mcc_th(≈0.06). F1 기준(≈0.13)보다 낮음.
#  ③ 검증: 두 후보를 격리 test에 적용해 '어느 쪽이 일반화되나' 확인 (아래 출력).
#     결과 — MCC 기준(0.06)은 train 과적합. test에서 정밀도 붕괴로 MCC가 오히려 하락.
#            F1 기준(0.13)이 test MCC를 더 안정적으로 유지.
#  ④ 결론: 임계값에 유일한 정답은 없다. '무엇을 중시하나'에 따른 trade-off이다.
#     - 종합 성능(MCC) 안정 중시 → 0.13  (기본값으로 채택)
#     - 불량 유출(FN) 최소화 최우선 → 더 낮은 값(재현율↑, 정밀도↓)
#     현업에선 불량 유출 비용 vs 재검사 비용의 크기로 이 지점을 정한다.
#  ※ 엄밀히는 임계값 후보 선택도 validation set으로 해야 test 오염이 없다.
#    데이터가 작아 별도 validation을 못 뺀 한계가 있어, test는 최종 확인에만 사용.
THRESH = f1_th   # 기본값: test MCC가 더 안정적인 F1 기준 임계값(≈0.13)
print(f"=== 최종 임계값: {THRESH:.3f} (기본값 채택) ===")
print(f"  MCC 기준({mcc_th:.3f})은 train 과적합으로 test MCC 하락 → F1 기준 채택.")
print("  임계값은 정답이 아니라 trade-off: 유출 최소화 중시 시 더 낮은 값으로 조정.")

pred_te = (proba_te >= THRESH).astype(int)

# (2.2) [검증 근거] 두 후보 임계값의 test 성능 — 위 ③ 결론의 데이터 근거
#   (test는 후보 비교·확인에만 사용. 이 표를 보고 새 임계값을 만들지 않음.)
print("\n--- 임계값 후보별 test 성능 (trade-off 확인) ---")
for _t, _name in [(f1_th, 'F1 기준(0.13, 채택)'), (mcc_th, 'MCC 기준(0.06)')]:
    _p = (proba_te >= _t).astype(int)
    print(f"  [{_name}] MCC {_mcc(y_test, _p):.3f} | 재현율 {recall_score(y_test, _p):.3f} "
          f"| 정밀도 {precision_score(y_test, _p):.3f} | F1 {f1_score(y_test, _p):.3f}")
print("  → MCC로 최적화한 0.06이 test에선 MCC 더 낮음(과적합). '지표 최적화 ≠ 일반화'.")

# (2.5) [before/after] 임계값 조정의 효과 — 기본 0.5 vs 조정값
# 첫 시도: 임계값을 손대지 않으면(기본 0.5) 불균형 때문에 불량을 거의 못 잡는다.
#          이 'before'를 직접 찍어, 임계값 조정이 왜 필요했는지를 숫자로 보여준다.
from sklearn.metrics import confusion_matrix as _cm
pred_default = (proba_te >= 0.5).astype(int)        # 조정 안 한 날것
print("=== [before] 기본 임계값 0.5 (조정 안 함) ===")
print(f"  confusion matrix:\n{_cm(y_test, pred_default)}")
n_caught = int(((pred_default == 1) & (y_test == 1)).sum())
n_fail = int((y_test == 1).sum())
print(f"  실제 불량 {n_fail}개 중 {n_caught}개 탐지 "
      f"| 재현율 {recall_score(y_test, pred_default):.3f} "
      f"| F1 {f1_score(y_test, pred_default):.3f}")
print(f"  → 기본값으로는 불량을 거의 못 잡음(불균형 함정). 임계값 조정 필요.")

print(f"\n=== [after] 조정 임계값 {THRESH:.3f} (train에서 결정) ===")
print(f"F1     {f1_score(y_test, pred_te):.3f}")
print(f"재현율 {recall_score(y_test, pred_te):.3f}  (실제 불량 중 잡은 비율)")
print(f"정밀도 {precision_score(y_test, pred_te):.3f}  (불량이라 한 것 중 맞은 비율)")
print("=" * 38)

# %% 13.5 [지표 보강] MCC + G-mean + 특이도 (논문 근거 지표)
# ── 지표 선택 논리 (왜 이 지표인가) ──
#  정확도   → 불균형에서 무의미. 다 정상이라 찍어도 93% (TN에 속음).        [버림]
#  F1       → 정밀도·재현율의 조화. 단 TN(정상 판별)을 무시함.            [주력이나 한계]
#  MCC      → confusion matrix 네 칸 모두 반영 + 불균형에 강건.          [★핵심]
#             (Chicco & Jurman 2020: 불균형에선 MCC가 F1·정확도보다 신뢰)
#  G-mean   → 재현율×특이도의 기하평균. 한쪽만 높으면 급락 → 꼼수 방지.    [보조]
#  재현율    → FN(불량 유출)을 직접 줄임. 후공정 특성상 1순위.            [목적 직결]
#  ROC-AUC  → 임계값 무관 '순위 능력'. 단 불균형서 낙관 편향 가능.        [참고만]
#  PR-AUC   → 소수클래스(불량)에 민감 → ROC와 교차 확인용.              [참고]
# 핵심: 지표를 '유행'이 아니라 '목적(불량 탐지·불균형)'에 맞춰 골랐다.
from sklearn.metrics import (confusion_matrix, matthews_corrcoef,
                             roc_auc_score, average_precision_score)

def full_metrics(y_true, y_pred, label="test"):
    cm = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm[0][0], cm[0][1], cm[1][0], cm[1][1]
    sensitivity = TP / (TP + FN + 1e-9)          # 재현율(불량 탐지율)
    specificity = TN / (FP + TN + 1e-9)          # 특이도(정상 판별율)
    g_mean = np.sqrt(sensitivity * specificity)
    mcc = matthews_corrcoef(y_true, y_pred)
    print(f"[{label}] 재현율 {sensitivity:.3f} | 특이도 {specificity:.3f} "
          f"| G-mean {g_mean:.3f} | MCC {mcc:.3f}")
    return {'재현율': sensitivity, '특이도': specificity, 'G_mean': g_mean, 'MCC': mcc}

print("\n=== 보강 지표 (격리 test) ===")
full_metrics(y_test, pred_te, "RF-final")
# 해석: MCC는 -1~1, 0이 무작위. 불균형이라 F1보다 보수적으로(낮게) 나오는 게 정상.
#       SECOM은 익명·소량·불균형이라 MCC 0.2~0.3대가 현실적 천장(공개 연구들도 유사).

# 참고 지표: ROC-AUC / PR-AUC (확률값으로 계산 — 라벨 아님에 주의)
roc = roc_auc_score(y_test, proba_te)
pr = average_precision_score(y_test, proba_te)
print(f"[참고] ROC-AUC {roc:.3f} | PR-AUC {pr:.3f}")
print("  주력은 MCC·재현율. ROC/PR은 불균형서 해석 주의하며 교차 확인용으로만 봄.")

# %% 13.6 [업무 자동화] 위험 로트 자동 스크리닝
# 목적: 현장 엔지니어가 매일 수백~수천 로트를 다 보지 않고, 모델이 위험 로트를
#       우선순위로 자동 선별 → 점검 대상을 좁혀 검토 시간 절감.
# 주의(데이터 한계): SECOM은 2008년 정적 데이터라 실시간 신규 로트는 없음.
#       격리한 test는 '모델이 학습에 쓰지 않은 데이터'이므로, 실전 신규 로트와
#       동일한 입장. 따라서 test를 '새로 들어온 로트'로 가정해 시연한다.
#       (구조는 실전 그대로 — 현장에선 이 함수에 신규 로트를 넣기만 하면 됨)

def screen_new_lots(new_data, top_n=20):
    """신규 로트 데이터를 받아 점검 우선순위를 자동 출력.
       실전: new_data 자리에 매일의 신규 로트를 넣으면 됨."""
    X = new_data[top_cols]
    proba = final_pipe.predict_proba(X)[:, 1]
    result = pd.DataFrame({'불량확률': proba}, index=X.index)
    result['판정'] = np.where(result['불량확률'] >= THRESH, '⚠ 점검', '정상')
    위험 = result[result['판정'] == '⚠ 점검'].sort_values('불량확률', ascending=False)
    print(f"[자동 스크리닝] 전체 {len(X)}개 중 {len(위험)}개 점검 권장 "
          f"(임계값 {THRESH:.2f})")
    print(f"  → 엔지니어는 전체가 아닌 상위 {min(top_n, len(위험))}개에 우선 집중")
    return 위험.head(top_n)

print("\n=== 위험 로트 자동 선별 (test를 신규 로트로 가정) ===")
위험로트 = screen_new_lots(X_test, top_n=20)
print(위험로트)

# (선택) 신규 로트가 '매일 순차 입고'되는 흐름 시뮬레이션
# for 일차, 배치 in enumerate(np.array_split(X_test, 5)):
#     print(f"\n--- {일차+1}일차 입고 ---")
#     screen_new_lots(배치, top_n=5)

# %% 13.7 [의사결정 보조] 임계값 트레이드오프 표
# 목적: 임계값을 낮추면 재현율↑(불량 더 많이 잡음)·정밀도↓(헛발질 증가).
#       '놓침 비용 vs 헛발질 비용'에 따라 운영 임계값을 조건부로 제시.
# 현업 연결: 불량을 고객사로 유출하는 비용이 크면 재현율 우선(임계값↓),
#            정상품 폐기·재검사 비용이 크면 정밀도 우선(임계값↑).
rows = []
for t in [0.10, 0.15, 0.20, 0.30, 0.40, 0.50]:
    pred_t = (proba_te >= t).astype(int)
    if pred_t.sum() == 0:   # 아무것도 안 잡으면 정밀도 정의 불가
        continue
    rows.append({
        '임계값': t,
        '재현율': round(recall_score(y_test, pred_t), 3),
        '정밀도': round(precision_score(y_test, pred_t), 3),
        'F1': round(f1_score(y_test, pred_t), 3),
        '점검대상수': int(pred_t.sum()),
    })
tradeoff = pd.DataFrame(rows)
print("\n=== 임계값 트레이드오프 (격리 test) ===")
print(tradeoff.to_string(index=False))
print("해석: 임계값↓ → 재현율↑·정밀도↓·점검대상↑ (놓침은 줄지만 헛발질 늘어남)")
print("      운영값은 '놓침 비용 vs 헛발질 비용'으로 현업과 함께 확정.")

# %% 13.8 [원인 분석] SHAP — 어떤 센서가 불량을 유발하는가
# 목적: '위험 로트 선별'에서 멈추지 않고, "왜 위험한가"를 센서 단위로 추적한다.
#   글로벌: 전체적으로 불량을 가장 많이 유발하는 센서 + 그 방향(값↑이 불량↑인지)
#   로컬:   개별 위험 로트가 '어떤 센서 때문에' 위험한지
# 누수 차단: 해석용 모델도 'train만'으로 학습하고, SHAP 값은 'test'에서 계산한다.
#   (메인 성능 평가는 상위50 모델 그대로. 여기는 원인 해석 전용 별도 트랙.)
# 변수 범위: 상위50으로 미리 거른 selection bias를 피하려고, 전처리만 끝난
#   전체 컬럼(약 250개)으로 학습 → "걸러지지 않은 상태에서의 원인"을 본다.
try:
    import shap

    # (1) 해석용 모델: 전처리 후 전체 컬럼, train만으로 학습
    cause_cols = X_train.columns.tolist()        # 전처리 후 전체(약 250개)
    Xtr_full = X_train.fillna(X_train.median())  # 트리모델용 단순 대치(train 기준)
    Xte_full = X_test.fillna(X_train.median())   # test도 'train의' median으로(누수 X)
    cause_rf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                      random_state=42, n_jobs=-1)
    cause_rf.fit(Xtr_full[cause_cols], y_train)

    # (2) SHAP 값 계산 (test에서). TreeExplainer는 트리모델 전용·정확.
    print("\n[SHAP] 계산 중... (수백 트리 × test 표본, 수 분 소요 가능)")
    explainer = shap.TreeExplainer(cause_rf)
    sv = explainer.shap_values(Xte_full[cause_cols])
    # 이진분류는 보통 [정상, 불량] 두 배열 → 불량(class=1)만 사용
    sv_fail = sv[1] if isinstance(sv, list) else sv
    if sv_fail.ndim == 3:            # (샘플, 피처, 클래스) 형태면 불량 클래스 선택
        sv_fail = sv_fail[:, :, 1]

    # (3) 글로벌: 평균 |SHAP|로 '불량 유발 기여' 큰 센서 순위 + 방향
    mean_abs = np.abs(sv_fail).mean(axis=0)             # 기여 크기
    # 방향: 센서값과 SHAP의 상관 부호. +면 '값↑일 때 불량↑', -면 '값↑일 때 불량↓'
    direction = []
    for i, c in enumerate(cause_cols):
        x = Xte_full[c].values
        s = sv_fail[:, i]
        if np.std(x) < 1e-9 or np.std(s) < 1e-9:
            direction.append(0.0)
        else:
            direction.append(np.corrcoef(x, s)[0, 1])
    glob = pd.DataFrame({
        '센서': cause_cols,
        '기여도(평균|SHAP|)': mean_abs,
        '방향': ['값↑→불량↑' if d > 0.1 else ('값↑→불량↓' if d < -0.1 else '불명확')
                for d in direction],
    }).sort_values('기여도(평균|SHAP|)', ascending=False)

    print("\n=== [글로벌] 불량을 가장 많이 유발하는 센서 TOP 10 ===")
    print(glob.head(10).to_string(index=False))
    print("해석: 익명 데이터라 센서의 물리적 정체는 알 수 없음.")
    print("      현업에선 이 센서 ID를 공정 파라미터 태그와 매핑해 장비·스텝을 역추적.")

    # (4) 로컬: 위험 로트 상위 몇 개가 '어떤 센서 때문에' 위험한지
    proba_full = cause_rf.predict_proba(Xte_full[cause_cols])[:, 1]
    risk_idx = np.argsort(proba_full)[::-1][:3]   # 불량확률 상위 3개 로트
    print("\n=== [로컬] 위험 로트별 주요 원인 센서 TOP 3 ===")
    for r in risk_idx:
        lot = Xte_full.index[r]
        contrib = pd.Series(sv_fail[r], index=cause_cols)
        top3 = contrib.sort_values(ascending=False).head(3)
        items = ", ".join(f"{c}(+{v:.3f})" for c, v in top3.items())
        print(f"  로트 {lot} (불량확률 {proba_full[r]:.3f}): {items}")
    print("해석: 각 로트의 불량 확률을 끌어올린 센서를 분해 → 점검 우선 포인트로 활용.")

    # (5) [조치 제안] 핵심 센서의 '임계값' 탐색 — 어느 값을 넘으면 불량 위험이 급증하나
    # 논문(DATA ANALYTICS 2022)은 F59가 특정 값을 넘으면 SHAP·불량률이 오르는
    #   '임계값'을 보고했다. 그 값은 논문 모델 기준이므로, 여기서는 '내 데이터'에서
    #   직접 임계값을 찾는다. SHAP이 양(+)으로 전환되는 센서값 = 불량 기여 시작점.
    # 활용: 그 값을 모니터링 알람 기준으로 제안. (PM·파라미터 조정은 센서 정체 확인 후)
    print("\n=== [조치 제안] 핵심 센서 임계값 + 불량률 비교 ===")
    y_te = y_test.values if hasattr(y_test, 'values') else np.asarray(y_test)
    for c in ['59', '103', '64']:
        if c not in cause_cols:
            print(f"  센서 {c}: (전처리 단계에서 제거됨 — 건너뜀)")
            continue
        xvals = Xte_full[c].values
        svals = sv_fail[:, cause_cols.index(c)]
        # SHAP이 양수인(불량 쪽으로 민) 구간의 '센서값 최소치' ≈ 임계값 시작점
        pos = xvals[svals > 0]
        if len(pos) == 0:
            print(f"  센서 {c}: 불량 기여(양의 SHAP) 구간 없음 — 건너뜀")
            continue
        thr = np.percentile(pos, 10)   # 양의 SHAP 구간 하단부 = 위험 시작 근처
        above = y_te[xvals >= thr]
        below = y_te[xvals < thr]
        r_above = above.mean() if len(above) else 0
        r_below = below.mean() if len(below) else 0
        print(f"  센서 {c}: 위험 시작 임계값 ≈ {thr:.3f}")
        print(f"    임계값 이상 로트 불량률 {r_above:.1%} (n={len(above)}) "
              f"| 미만 {r_below:.1%} (n={len(below)})")
    print("해석: 임계값 이상에서 불량률이 높으면, 그 값을 모니터링 알람 기준으로 제안 가능.")
    print("      센서의 물리적 정체 확인 후 PM·파라미터 조정 등 구체 조치로 연결.")

except ImportError:
    print("\n[SHAP 분석 건너뜀] shap 미설치.")
    print("  설치: import sys; !{sys.executable} -m pip install shap")
    print("  설치 후 numpy 충돌 시: !{sys.executable} -m pip install 'numpy<2' → 커널 재시작")

# %% 14. 정리
# - 전처리: 상수/고결측/중복 제거 → median 대치 (근거: 왜도)
# - 불균형: SMOTE 재검증 후 판단, 임계값 조정이 핵심
#   · 임계값을 MCC로 최적화 시도(0.06) → test에서 과적합 확인(MCC 하락) → F1 기준(0.13) 채택
#   · 교훈: '지표로 최적화 ≠ 일반화'. 임계값은 정답이 아니라 목적에 따른 trade-off.
# - 피처: RF 중요도 상위 50개 (차원 축소 = 핵심 레버)
# - 모델: RF 유지 (교체 무익)
# - 평가: test 처음에 격리 → 끝에 한 번 → 일반화 성능 정직하게 보고
#   · 지표: MCC(핵심)·재현율 + G-mean(보조) + ROC/PR-AUC(참고). 목적에 맞춰 선택.
# - 원인 분석(SHAP): 센서 59·103·64가 불량 유발 (선행연구 F59·F64·F103과 일치)
#   · 조치: 센서 59가 ≈5.66 초과 시 불량률 8배 → 모니터링 알람 기준 제안
# - 업무 적용: 격리 test를 신규 로트로 가정 → 위험 로트 자동 스크리닝 시연
# - 의사결정: 임계값 트레이드오프 표 → 놓침/헛발질 비용으로 운영값 조건부 제시
#
# 결과: MCC 0.373 (공개 분석 0.2 안팎보다 누수 없이 높은 편. 단 절대값이 높진 않음).
# 한계: SECOM은 익명·소량·불균형이라 성능 천장이 낮음. 정적 데이터라 실시간 입력 없음.
#
# ── 한 구조가 두 축을 동시에 충족 ──
#  · 축2(원인 분석): train만으로 결정 → 누수 없이 센서 원인 추적·검증
#  · 축1(업무 적용): 격리 test = 신규 로트 → 자동 스크리닝·임계값 알람 시연
#  → "맨 처음 test 격리" 하나로 분석의 정직성과 실전 적용을 모두 확보.
#
# ── 개선 경로: 공정 도메인과 결합할 때 ──
#  MCC가 높지 않은 핵심 원인 = 익명이라 도메인 지식을 못 씀. 현업 결합 시:
#   · 대치 방법 재검토 (연속공정→ffill / 미진행→0·플래그) ← median은 통계적 선택일 뿐
#   · 센서 정체 기반 피처 가공 (비선형 변환, 공정 변수 비율, 결측의 설비상태 의미화)
#   · 공정 흐름 기반 파생 변수, 공정 특성에 맞는 모델 선택
#  → 데이터로 원인을 좁힌 이 분석 위에 공정 지식이 더해질 때 비로소 완성됨.

# %% [markdown]
# ## 다음 단계 (선택)
# - 재현율 더 중시(불량 유출 비용↑) 시 임계값 낮춰 재현율↑/정밀도↓ 트레이드오프 표 작성
# - 최종 임계값은 '놓침 비용 vs 헛발질 비용' 현업 데이터로 확정

전체: (1567, 590) | 불량 비율: 0.066
train 불량 0.066 | test 불량 0.067  (동일해야 정상)
① 상수/전부NaN: 116개
[결측의 행 방향 분포]
  로트별 결측률 평균 0.046 | 최대 0.258 | 최소 0.007
  결측률 30% 초과 로트: 0개 / 1253개
  상위 5% 로트의 결측률: 평균 0.128 (전체 평균 대비 2.8배)
  결측 여부가 불량과 연관 의심되는 열: 0개
해석: 군집성↑ 또는 연관 열↑ 이면 단순 제거 대신 '결측 플래그' 보존을 검토.
      (이번 데이터는 익명화라 정체 불명 — 패턴만 확인하고 제거 결정에 반영)
② 고결측(>50%): 24개
③ 고상관(>0.9): 200개
피처: 590 → 250  (총 340개 제거)
왜도 |>1 (치우침): 126 / 211개  → median 근거
자기상관 평균 -0.007, |0.5| 초과 0개 → 시계열성 약함/혼재, ffill 기각 · median 채택
[전체 피처]
RF(전체)             | 최적임계 0.12 | F1 0.259 | 재현율 0.301 | 정밀도 0.227
[상위 K개]
RF(상위20)           | 최적임계 0.20 | F1 0.287 | 재현율 0.301 | 정밀도 0.275
RF(상위50)           | 최적임계 0.13 | F1 0.295 | 재현율 0.434 | 정밀도 0.224
RF(상위100)          | 최적임계 0.13 | F1 0.309 | 재현율 0.386 | 정밀도 0.258
[모델 비교 — 상위 50개]
RandomForest       | 최적임계 0.13 | F1 0.295 | 재현율 0.434 | 정밀도 0.224
LogisticReg        | 최적임계 0.64 | F1 0.236 | 재현율 0.446 | 정밀도 0.161
[SMOTE 재검증 — 상위 50개]
기본                 | 최적임계 0.10 | F1 0.254 | 재현율 0.